# Visualization
**Topic:** Scientific Python

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


---
## What you'll explore

By the end of this demo you will be able to:

- **Select** the appropriate chart type for a given data question
- **Describe** how the grammar of graphics maps data columns to visual properties (x, y, color, size)
- **Interpret** a multi-panel EDA dashboard and read the story it tells about a dataset

---
## How we got here

In *15: Data Cleaning* we made the data trustworthy. In *08: Data Visualization for Statistics* (in the statistics series) we used visualization to understand distributions and correlations. This notebook extends that into the full matplotlib and seaborn API and connects the chart choices directly to the data science questions that each chart type is designed to answer.

---
## Why this matters for data science

Visualization is both how you understand your data and how you communicate your findings. Before training any model, you should be able to produce a histogram of every feature, a scatter plot of key relationships, a box plot comparing groups, and a heatmap of correlations. After training, you need a residual plot, a confusion matrix heatmap, and an ROC curve. These are not optional extras, they are how you catch problems that summary statistics miss.

---
## Try it yourself

In [ ]:
# ▶ Run this cell and observe the output.
# Then try changing the column passed to x= and running again.

np.random.seed(16)
n = 300
df = pd.DataFrame({
    "age":        np.random.randint(22, 60, n),
    "salary":     np.random.lognormal(10.8, 0.4, n),
    "department": np.random.choice(["Engineering", "Marketing", "Operations"], n),
    "tenure":     np.random.exponential(4, n),
})

fig, ax = plt.subplots(figsize=(8, 4))
sns.histplot(df, x="salary", hue="department", bins=30, kde=True, ax=ax)
ax.set_title("Salary Distribution by Department")
ax.set_xlabel("Annual Salary ($)")
plt.tight_layout()
plt.show()


In [ ]:
# ✏️ Your turn — modify this code:
# 1. Change sns.boxplot to sns.violinplot and compare the output
# 2. Add hue="department" to sns.scatterplot to color points by group
# 3. What does the scatter plot reveal that the box plot hides?

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.boxplot(data=df, x="department", y="salary", ax=axes[0])
axes[0].set_title("Salary by Department (box)")
axes[0].set_xlabel("Department")
axes[0].set_ylabel("Annual Salary ($)")

sns.scatterplot(data=df, x="tenure", y="salary", ax=axes[1])
axes[1].set_title("Salary vs Tenure (scatter)")
axes[1].set_xlabel("Tenure (years)")
axes[1].set_ylabel("Annual Salary ($)")

plt.tight_layout()
plt.show()


In [ ]:
# 🎯 Challenge:
# Build the four-chart EDA sequence for the df dataset above:
# 1. sns.histplot of salary (one panel)
# 2. sns.scatterplot of salary vs tenure, with hue="department"
# 3. sns.boxplot of salary by department
# 4. sns.heatmap of df[["age","salary","tenure"]].corr(), annot=True
# Arrange all four charts in a 2x2 grid:
#   fig, axes = plt.subplots(2, 2, figsize=(12, 8))

# Your code here:


---
## What's happening?

Every chart maps one or more data columns to visual channels: position (x, y), color, size, shape, or opacity. Choosing the right mapping for the question you are asking is what makes a chart informative instead of just decorative.

| Question | Best chart | Key mapping |
|----------|-----------|------------|
| Distribution of one variable | Histogram or KDE | x = variable, y = count/density |
| Relationship between two numeric variables | Scatter plot | x = variable 1, y = variable 2 |
| Distribution across groups | Box or violin | x = group, y = variable |
| Correlation across many variables | Heatmap | color = correlation value |
| Trend over time | Line chart | x = date, y = metric |
| Part-to-whole breakdown | Bar chart | x = category, y = value |

```python
# seaborn: concise chart creation
fig, ax = plt.subplots(figsize=(8, 4))
sns.scatterplot(df, x="age", y="salary", hue="department", ax=ax)
ax.set_title("Salary vs Age by Department")
plt.tight_layout()
plt.show()
```

### The EDA sequence

Every dataset deserves this four-chart sequence: (1) histogram of each feature to check shape and outliers, (2) scatter plot of target vs each numeric feature to assess relationships, (3) box plot of target grouped by each categorical feature, (4) correlation heatmap of all numeric features.

Return to the exercises above and work through this sequence on the pre-loaded dataset.

---
## A direct example: mapping three columns to one chart

Eight students, three columns, one scatter plot. Seaborn maps `hours_studied` to x, `score` to y, and `passed` to color — each column becomes a visual channel.

- **Notice:** The upward trend is immediately visible without any calculation — visualization surfaces the pattern before any model is built
- **Notice:** Adding `hue="passed"` maps a third column to color at no extra cost; this is the grammar of graphics in action
- **Notice:** A bar chart of average scores would also show the trend, but a scatter plot shows the individual variation that an average hides

In [ ]:
df_students = pd.DataFrame({
    "hours_studied": [2, 5, 1, 8, 3, 6, 4, 7],
    "score":         [58, 82, 45, 94, 65, 88, 72, 91],
    "passed":        ["No", "Yes", "No", "Yes", "Yes", "Yes", "Yes", "Yes"],
})

fig, ax = plt.subplots(figsize=(7, 4))
sns.scatterplot(data=df_students, x="hours_studied", y="score",
                hue="passed", palette={"Yes": "#4C72B0", "No": "#E45756"},
                s=100, ax=ax)
ax.set_title("Hours studied vs exam score", fontsize=13)
ax.set_xlabel("Hours Studied")
ax.set_ylabel("Score")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

---
## Real-world example: EDA dashboard for a customer churn dataset

The chart below shows a two-panel EDA snapshot of a simulated customer churn dataset: the distribution of monthly charges split by churn status, and the churn rate by contract type. These two charts answer the two most important questions in churn analysis before any model is built.

Notice:

- **Notice:** Customers who churned have a higher concentration of high monthly charges, which suggests price sensitivity is a churn driver worth modeling
- **Notice:** Month-to-month contract customers churn at a dramatically higher rate than annual contract customers, making contract type one of the strongest predictors even before any feature engineering
- **Notice:** Both insights emerge from simple visualization; no model is needed to surface them, which is why EDA always comes before modeling

> **Discussion question:** The monthly charge distribution for churned customers is right-skewed relative to retained customers. Before encoding monthly charge as a raw numeric feature in a logistic regression model, what transformation would you consider and why?

In [ ]:
np.random.seed(55)

# ── Simulated customer churn dataset ─────────────────────────────────────────
n             = 600
contract_type = np.random.choice(["Month-to-month", "One year", "Two year"], n,
                                  p=[0.55, 0.25, 0.20])
churn_prob    = {"Month-to-month": 0.45, "One year": 0.12, "Two year": 0.05}
churned       = np.array([np.random.rand() < churn_prob[c] for c in contract_type])

monthly_charge = np.where(
    churned,
    np.random.lognormal(4.1, 0.4, n),
    np.random.lognormal(3.7, 0.5, n),
)
df_churn = pd.DataFrame({
    "contract":       contract_type,
    "monthly_charge": monthly_charge,
    "churned":        churned,
})

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for status, color, label in [(True, "#E45756", "Churned"), (False, "#4C72B0", "Retained")]:
    subset = df_churn.loc[df_churn["churned"] == status, "monthly_charge"]
    axes[0].hist(subset, bins=30, density=True, color=color, alpha=0.6, label=label)
axes[0].set_title("Monthly Charge by Churn Status")
axes[0].set_xlabel("Monthly Charge ($)")
axes[0].set_ylabel("Probability Density")
axes[0].legend()
axes[0].spines[["top", "right"]].set_visible(False)

churn_rate = df_churn.groupby("contract")["churned"].mean() * 100
contracts  = ["Month-to-month", "One year", "Two year"]
axes[1].bar(contracts, [churn_rate[c] for c in contracts],
            color=["#E45756", "#4C72B0", "#55A868"])
axes[1].set_title("Churn Rate by Contract Type")
axes[1].set_xlabel("Contract Type")
axes[1].set_ylabel("Churn Rate (%)")
axes[1].spines[["top", "right"]].set_visible(False)

fig.suptitle("Customer Churn EDA Dashboard", fontsize=14)
plt.tight_layout()
plt.show()


### Chart type quick-reference for data science

| Goal | Chart | Matplotlib/Seaborn call |
|------|-------|-------------------------|
| Distribution of one variable | Histogram | `sns.histplot(df, x="col")` |
| Two numeric variables | Scatter | `sns.scatterplot(df, x="a", y="b", hue="group")` |
| Compare groups | Box or violin | `sns.boxplot(df, x="group", y="metric")` |
| Correlation matrix | Heatmap | `sns.heatmap(df.corr(), annot=True)` |
| Trend over time | Line | `sns.lineplot(df, x="date", y="metric")` |
| Model residuals | Scatter | `plt.scatter(y_pred, residuals)`

---
## Key takeaway

> **The right chart type is determined by the question you are asking, not by personal preference; choosing wrongly does not produce an error — it produces a misleading picture that leads to wrong conclusions.**

---
*Next up: Files & APIs — how to get data into your notebook from CSV files, JSON files, and live REST APIs*